# Productos vendidos
Notebook que lee una carpeta con facturas emitidas en formato pdf y guarda la informacion sobre el producto vendido.
Esto deberia coincidir con una parte de los movimientos bancarios asi como tambien los movimientos de facturas emitidas.
Pero aca el interes esta en que y cuanto se vendio, y no tanto en el costo ya que eso esta en otro lado.
Se duplicaria la info, pero esta bueno considerarlo como una validacion.

In [3]:
#Librerias 
import pandas as pd
import numpy as np
import os
import matplotlib.pyplot as plt
import re
import pdfplumber
print("Librerias ok")

Librerias ok


In [ ]:
#Pruebo con un solo archivo
ruta = "D:/Josefina/Proyectos/Datascience/Data_Analytics/Contabilidad_Empresa/data/raw/facturas/30715431013_006_00001_00003988.pdf"


In [ ]:
#Algunos atributos de pdf
with pdfplumber.open(ruta) as pdf:
    print("Cantidad de páginas:", len(pdf.pages))
    print("Metadatos:", pdf.metadata)
    pagina = pdf.pages[0]
    print("Ancho:", pagina.width)
    print("Alto:", pagina.height)
    print("Caracteres:", len(pagina.chars))
    print("Palabras:", len(pagina.extract_words()))
    print("Imágenes:", len(pagina.images))
    print("Líneas:", len(pagina.lines))


Cantidad de páginas: 3
Metadatos: {'ModDate': "D:20260731163306-03'00'", 'Creator': 'ARCA', 'CreationDate': "D:20260731163306-03'00'", 'Producer': 'iText 2.1.7 by 1T3XT', 'Author': 'ARCA'}
Ancho: 595
Alto: 842
Caracteres: 960
Palabras: 156
Imágenes: 2
Líneas: 32


In [41]:
with pdfplumber.open(ruta) as pdf:
    texto = pdf.pages[0].extract_text()
texto

'ORIGINAL\nB\nFACTURA\nISLA ECOLOGICA S.A.\nCOD. 006\nPunto de Venta: 00001 Comp. Nro: 00003910\nRazón Social: ISLA ECOLOGICA S.A. Fecha de Emisión: 01/07/2026\nDomicilio Comercial: Av. Sabattini 1877 - Barrio Maipu, Córdoba CUIT: 30715431013\nIngresos Brutos: .\nCondición frente al IVA: IVA Responsable Inscripto Fecha de Inicio de Actividades: 01/11/2016\nDoc.: - Apellido y Nombre / Razón Social:\nCondición frente al IVA: Consumidor Final Domicilio:\nCondición de venta: Contado / Cuenta Corriente / Otra\nCódigo Producto / Servicio Cantidad U. Medida Precio Unit. % Bonif Imp. Bonif. Subtotal\nMadera 1,00 unidades 19800,00 0,00 0,00 19800,00\nSubtotal: $ 19800,00\nImporte Otros Tributos: $ 0,00\nImporte Total: $ 19800,00\nRégimen de Transparencia Fiscal al Consumidor (Ley 27.743)\nIVA Contenido: $ 3436,36\nPág. 1/1 CAE N°: 86272145658302\nFecha de Vto. de CAE: 11/07/2026\nComprobante Autorizado\nEsta Agencia no se responsabiliza por los datos ingresados en el detalle de la operación'

In [23]:
def extraer_producto_factura(ruta_pdf):
    # 1. Leer PDF
    with pdfplumber.open(ruta_pdf) as pdf:
        texto = pdf.pages[0].extract_text()
    lineas = texto.split("\n")

    # 2. Patrones
    patron_encabezado = re.compile(r"Punto de Venta:\s*(\d+)\s+Comp\.\s*Nro:\s*(\d+)")
    patron_fecha = re.compile(r"Fecha de Emisión:\s*(\d{2}/\d{2}/\d{4})")
    patron_cliente_doc = re.compile(r"Doc\.:\s*(.*?)\s+Apellido y Nombre / Razón Social:")
    patron_cliente_razon_social = re.compile(r"Apellido y Nombre / Razón Social:\s*(.*)")
    patron_cliente_condiva = re.compile(r"Condición frente al IVA:\s*(.*?)\s+Domicilio:")
    patron_cliente_domicilio = re.compile(r"Domicilio:\s*(.*)")
    patron_cliente_condventa = re.compile(r"Condición de venta:\s*(.*)")
    patron_otros_tributos = re.compile(r"Importe Otros Tributos:\s*\$\s*([\d.,]+)")
    patron_importe_total = re.compile(r"Importe Total:\s*\$\s*([\d.,]+)")
    patron_IVAcontenido = re.compile(r"IVA Contenido:\s*\$\s*([\d.,]+)")
    patron_producto = re.compile(
        r"(.+?)\s+"                 # producto
        r"(\d+,\d{2})\s+"           # cantidad
        r"(\w+)\s+"                 # unidad
        r"(\d+,\d{2})\s+"           # precio unitario
        r"(\d+,\d{2})\s+"           # bonificación
        r"(\d+,\d{2})\s+"           # importe bonificación
        r"(\d+,\d{2})$"             # subtotal
    )

    # 3. Variables
    punto_venta = None
    comp_nro = None
    fecha = None
    cliente_doc = None
    razon_social = None
    cond_iva = None
    domicilio = None
    cond_venta = None
    otros_tributos = None
    inicio = None
    importeTotal = None
    IVA_cotenido = None

    # 4. Recorrer todas las lineas 1 sola vez
    for i, linea in enumerate(lineas):
        if punto_venta is None:
            match = patron_encabezado.search(linea)
            if match:
                punto_venta = match.group(1)
                comp_nro = match.group(2)
        if fecha is None:
            match = patron_fecha.search(linea)
            if match:
                fecha = match.group(1)
        if cliente_doc is None:
            match = patron_cliente_doc.search(linea)
            if match:
                cliente_doc = match.group(1).strip()
                if cliente_doc == "-":
                    cliente_doc = None
        if razon_social is None:
            match = patron_cliente_razon_social.search(linea)
            if match:
                razon_social = match.group(1).strip()
                if razon_social == "":
                    razon_social = None
        if cond_iva is None:
            match = patron_cliente_condiva.search(linea)
            if match:
                cond_iva = match.group(1).strip()
        if domicilio is None:
            match = patron_cliente_domicilio.search(linea)
            if match:
                domicilio = match.group(1).strip()
                if domicilio == "":
                    domicilio = None
        if cond_venta is None:
            match = patron_cliente_condventa.search(linea)
            if match:
                cond_venta = match.group(1).strip()
        if otros_tributos is None:
            match = patron_otros_tributos.search(linea)
            if match:
                otros_tributos = float(match.group(1).replace(".", "").replace(",", "."))
        if importeTotal is None:
                match = patron_importe_total.search(linea)
                if match:
                    importeTotal = float(match.group(1).replace(".", "").replace(",", "."))
        if IVA_cotenido is None:
                    match = patron_IVAcontenido.search(linea)
                    if match:
                        IVA_cotenido = float(match.group(1).replace(".", "").replace(",", "."))
        if inicio is None and "Código" in linea:
            inicio = i

    # 6. Extraer productos
    factura = []
    primera_fila = True

    for linea in lineas[inicio + 1:]:
        if "Subtotal:" in linea:
            break
        match = patron_producto.match(linea)
        if match:
            datos_generales = {
                "cliente_doc": cliente_doc,
                "fecha_emision": fecha,
                "punto_venta": punto_venta,
                "comp_nro": comp_nro,
                "razon_social": razon_social,
                "condicion_iva": cond_iva,
                "domicilio": domicilio,
                "condicion_venta": cond_venta,
                "producto": match.group(1).strip(),
                "cantidad": match.group(2),
                "unidad": match.group(3),
                "precio_unitario": float(match.group(4).replace(".", "").replace(",", ".")),
                "bonificacion": float(match.group(5).replace(".", "").replace(",", ".")),
                "importe_bonificacion": float(match.group(6).replace(".", "").replace(",", ".")),
                "subtotal": float(match.group(7).replace(".", "").replace(",", "."))}
            
            # Datos que pertenecen a la factura completa solo se agregan una vez
            if primera_fila:
                datos_generales["IVA_contenido"] = IVA_cotenido
                datos_generales["otros_tributos"] = otros_tributos
                datos_generales["importeTotal"] = importeTotal
                primera_fila = False
            else:
                datos_generales["IVA_contenido"] = None
                datos_generales["otros_tributos"] = None
                datos_generales["importeTotal"] = None
            factura.append(datos_generales)
    
    # 7. DataFrame
    df_factura = pd.DataFrame(factura)
    #df_factura["suma"] = df_factura["subtotal"] + df_factura["IVA_cotenido"]
    return df_factura

In [ ]:
#Pruebo la funcion
ruta = "D:/Josefina/Proyectos/Datascience/Data_Analytics/Contabilidad_Empresa/data/raw/facturas/30715431013_006_00001_00003988.pdf"
extraer_producto_factura(ruta)

In [ ]:
#Pruebo funcion donde estan todos los archivos
carpeta = "D:/Josefina/Proyectos/Datascience/Data_Analytics/Contabilidad_Empresa/data/raw/facturas/"
archivos = []
os.listdir(carpeta)
for archivo in os.listdir(carpeta):
    # Verificar que sea un PDF
    if archivo.lower().endswith(".pdf"):
        print(archivo)
        # Crear la ruta completa
        ruta_pdf = os.path.join(carpeta, archivo)
        # Aplicar funcion
        df = extraer_producto_factura(ruta_pdf)
         # Agregar el nombre del archivo como una nueva columna
        df["archivo_origen"] = os.path.splitext(archivo)[0]
        # Guardar el DataFrame en la lista
        archivos.append(df)
        # Unir todos los DataFrames
    df_final = pd.concat(archivos, ignore_index=True)

# 

30715431013_001_00001_00000339.pdf
30715431013_001_00001_00000340.pdf
30715431013_006_00001_00003908.pdf
30715431013_006_00001_00003909.pdf
30715431013_006_00001_00003910.pdf
30715431013_006_00001_00003911.pdf
30715431013_006_00001_00003912.pdf
30715431013_006_00001_00003913.pdf
30715431013_006_00001_00003914.pdf
30715431013_006_00001_00003915.pdf
30715431013_006_00001_00003916.pdf
30715431013_006_00001_00003917.pdf
30715431013_006_00001_00003918.pdf
30715431013_006_00001_00003919.pdf
30715431013_006_00001_00003920.pdf
30715431013_006_00001_00003921.pdf
30715431013_006_00001_00003922.pdf
30715431013_006_00001_00003923.pdf
30715431013_006_00001_00003924.pdf
30715431013_006_00001_00003925.pdf
30715431013_006_00001_00003927.pdf
30715431013_006_00001_00003928.pdf
30715431013_006_00001_00003929.pdf
30715431013_006_00001_00003930.pdf
30715431013_006_00001_00003931.pdf
30715431013_006_00001_00003932.pdf
30715431013_006_00001_00003933.pdf
30715431013_006_00001_00003934.pdf
30715431013_006_0000

In [ ]:
df_final.to_csv("D:/Josefina/Proyectos/Datascience/Data_Analytics/Contabilidad_Empresa/data/prueba.csv")